In [2]:
from collections import defaultdict
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader,random_split
import torch.optim as optim
random.seed(42)

In [3]:
if torch.cuda.is_available():
    print("cuda is available")

cuda is available


In [4]:
train1 = torch.load("./train_val_embedding_data/circuit_breakers_others.pt", map_location=torch.device('cpu'))
train2 = torch.load("./train_val_embedding_data/circuit_breakers_actorattack.pt", map_location=torch.device('cpu'))

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_26156\405173482.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train1 = torch.load("./train_val_embedding_data/circu

In [5]:
train1[0][0].keys()

dict_keys(['u', 'y', 'score', 'round'])

In [6]:
scores1=defaultdict(int)
scores2=defaultdict(int)
for conversation in train1:
    for turn in conversation:
        scores1[turn['score']]+=1

for conversation in train2:
    for turn in conversation:
        scores2[turn['score']]+=1

print("Score distribution:")
print(sorted(scores1.items()))
print(sorted(scores2.items()))

Score distribution:
[(1, 4629), (2, 2442), (3, 6199), (4, 1206), (5, 1724)]
[(1, 2719), (2, 2170), (3, 4848), (4, 2149), (5, 893)]


In [7]:
convo1_len=defaultdict(int)
convo2_len=defaultdict(int)
for conversation in train1:
    convo1_len[len(conversation)]+=1
for conversation in train2:
    convo2_len[len(conversation)]+=1

print("Conversation length distribution:")
print(sorted(convo1_len.items()))
print(sorted(convo2_len.items()))

Conversation length distribution:
[(1, 583), (2, 151), (3, 179), (4, 226), (5, 201), (6, 134), (7, 143), (8, 1383)]
[(1, 6), (2, 5), (3, 4), (4, 3), (5, 1115), (6, 1194)]


In [8]:
original_training_data=train1+train2
training_data = random.sample(original_training_data, len(original_training_data))

assert len(training_data)==len(train1)+len(train2)
assert training_data!=original_training_data
assert len(training_data)==len(original_training_data)

# model

In [16]:
class stateSpaceModel(nn.Module):
    def __init__(self,state_dim, input_dim, hidden_dim1,hidden_dim2, output_dim):
        super(stateSpaceModel, self).__init__()
        # F(x_{t-1},u_t)=>x_t (Transition model)
        self.Fxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, state_dim)
        )

        # G(x_t,u_t)=>z_t (Observation model)
        self.Gxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, output_dim)
        )
        
    def forward(self, x_prev, u):
        # gets xt
        ux_prev = torch.cat([u,x_prev], dim=-1)
        x_curr = self.Fxu(ux_prev)

        # gets zt
        ux_curr = torch.cat([u,x_curr], dim=-1)
        zt = self.Gxu(ux_curr)
        return x_curr, zt

In [17]:
state_dim = 768    # x_t
input_dim = 768    # u_t
output_dim = 768   # z_t
hidden_dim_ssm1 = 1200  
hidden_dim_ssm2 = 900  

model=stateSpaceModel(state_dim=state_dim, input_dim=input_dim, hidden_dim1=hidden_dim_ssm1, hidden_dim2=hidden_dim_ssm2, output_dim=output_dim)

In [19]:
import torch

checkpoint = torch.load("./ssm_model_exp5/models_best_ssm.pth")
model.load_state_dict(checkpoint["ssm"])
model.eval()

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_26156\475773191.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("./ssm_model_exp5/models_best

stateSpaceModel(
  (Fxu): Sequential(
    (0): Linear(in_features=1536, out_features=1200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1200, out_features=900, bias=True)
    (3): ReLU()
    (4): Linear(in_features=900, out_features=768, bias=True)
  )
  (Gxu): Sequential(
    (0): Linear(in_features=1536, out_features=1200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1200, out_features=900, bias=True)
    (3): ReLU()
    (4): Linear(in_features=900, out_features=768, bias=True)
  )
)

In [32]:
all_conversations=[]
with torch.no_grad():  # no gradients for inference
    for convo in training_data:
        single_conversation=[]
        x_prev = torch.zeros(state_dim)  
        for turn in convo:
            u = turn['u']  
            x_curr, zt = model(x_prev, u)
            single_conversation.append({"x_t":x_curr,
                                        "ut":u,
                                        "zt":zt,
                                        "score":turn['score'],
                                        "y":turn['y']})
            x_prev = x_curr
        all_conversations.append(single_conversation)


In [33]:
torch.save(all_conversations, "./train_data_with_context/all_conversations2.pt")